In [ ]:
    ############    #############   Testing strategy   #############   ##############   

 =>  Phase: P0 -- FDE Foundations & Engineering Baseline
 =>  Topic: 0.3 Engineering Standards
 =>  Week:  Week 1
 =>  Track: Core

 =>  Sections to fill in:
       1. Theory
       2. Diagram(s) / flowchart(s) (images/ folder)  -- only where the concept needs one
       3. Command / config reference (if applicable)
       4. Runnable code demo(s)
       5. Hands-on lab checklist
       6. Common pitfalls / notes


In [ ]:
    ############    #############   The Testing Pyramid   #############   ##############   

 =>  Not all tests are equally cheap. The pyramid shape is a budget: many fast/cheap unit
       tests, fewer slower integration tests, a handful of end-to-end tests covering only
       the most critical paths, and manual testing last of all.

 =>  An 'ice cream cone' (inverted pyramid -- mostly slow E2E tests, few unit tests) is a
       common anti-pattern: the test suite becomes slow, flaky, and everyone starts
       skipping it.


<img src="images/testing-pyramid.png" alt="The testing pyramid: unit tests at the base, then integration, end-to-end, and manual/exploratory at the top">

In [ ]:
import pytest

# ---- Unit test: pure logic, no I/O, no framework, milliseconds to run ----
def apply_discount(price: float, percent_off: float) -> float:
    if not 0 <= percent_off <= 100:
        raise ValueError("percent_off must be between 0 and 100")
    return round(price * (1 - percent_off / 100), 2)

def test_apply_discount_normal_case():
    assert apply_discount(100.0, 20) == 80.0

def test_apply_discount_zero_percent():
    assert apply_discount(50.0, 0) == 50.0

def test_apply_discount_rejects_invalid_percent():
    with pytest.raises(ValueError):
        apply_discount(50.0, 150)

# run these 3 tests directly (outside a real pytest session) to show pass/fail clearly
for test_fn in [test_apply_discount_normal_case, test_apply_discount_zero_percent,
                test_apply_discount_rejects_invalid_percent]:
    test_fn()
    print(f"PASSED: {test_fn.__name__}")


In [ ]:
 =>  In a real project these live in test_*.py files and run via 'pytest' -- shown
       inline here just to demonstrate the shape of a good unit test: one behavior per
       test, a clear assertion, no setup of a database/network/filesystem.

 =>  Notice all 3 tests together run in microseconds -- this is exactly why the pyramid
       puts the most tests here: they're cheap enough to run on every save.


In [ ]:
# ---- Integration-style test: exercises a real boundary (here, an in-memory 'repository') ----
class InMemoryOrderRepository:
    def __init__(self):
        self._orders: dict[int, dict] = {}
        self._next_id = 1

    def create(self, total: float) -> dict:
        order = {"id": self._next_id, "total": total}
        self._orders[self._next_id] = order
        self._next_id += 1
        return order

    def get(self, order_id: int) -> dict | None:
        return self._orders.get(order_id)

def test_create_then_get_order_round_trips():
    repo = InMemoryOrderRepository()
    created = repo.create(total=42.50)
    fetched = repo.get(created["id"])
    assert fetched == created

test_create_then_get_order_round_trips()
print("PASSED: test_create_then_get_order_round_trips")


In [ ]:
 =>  A REAL integration test would point this at an actual test Postgres database
       instead of an in-memory dict -- the point is testing that two real components (your
       code + the actual storage layer) work together correctly, which a unit test with a
       fake can't fully prove.


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Install pytest and move the tests above into a real test_pricing.py file; run
           'pytest -v' and read the actual output format.

 =>  [ ] Add one integration test against a real (Dockerized) Postgres instance using a
           test-specific database, and one E2E test using FastAPI's TestClient against a
           full app from Phase 0.2.

 =>  [ ] Audit your test suite's actual shape (count unit vs integration vs E2E tests) --
           does it look like a pyramid or an ice cream cone?


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Writing tests that assert on implementation details (private method call counts)
       instead of observable behavior -- these break on every harmless refactor.

 =>  No tests for error paths, only the happy path -- the ValueError case above is exactly
       as important to test as the normal case, often more so.

 =>  Flaky tests (pass sometimes, fail sometimes, e.g. due to real timing/network) left in
       the suite instead of fixed or removed -- teams learn to ignore red builds, which
       defeats the entire purpose of having tests.
